# GIN neighborhood perturbation analysis

This notebook runs the **neighborhood perturbation analysis** using
`compute_perturbation_influence_maps()` for **GINCurvature** models:

- **with** global features
- **without** global features

at depths:

- **2**
- **4**
- **6**

For each configuration, it:
1. loads and prepares the dataset using the updated metadata conventions,
2. trains a GIN model,
3. builds and samples validation ego-subgraphs,
4. computes perturbation influence maps,
5. plots the results,
6. saves the experiment outputs.


In [ ]:
import copy
import inspect
import json
import math
import pickle
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().parent
DATA_ROOT = PROJECT_ROOT / "training_data"

sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT    =", DATA_ROOT)
print("torch        =", torch.__version__)
print("cuda         =", torch.cuda.is_available())


In [ ]:
# Data and target settings
EXPERIMENT_GROUP = "ablation_analysis"
DATA_SUBDIR = "mean_curvature_smooth"
TARGET_INDICES = [0]
USE_GLOBAL_FEATURES = True
FEATURE_VARIANTS = {
    "with_global" if USE_GLOBAL_FEATURES else "no_global": USE_GLOBAL_FEATURES,
}

# Filtering and preprocessing settings
MISSING_COMPLEXITY_GROUP = {
    "dataset": "20251201",
    "timepoint": "day4p5",
    "fill_value": 2.1,
}
COMPLEXITY_THRESHOLD = 2.0
SPHERICITY_MAX = 0.92
SPHERICAL_MARKER_DIVERSITY_MIN = 0.5
INTERPOLATE_TARGET_OUTLIERS = True
OUTLIER_CLIP_QUANTILES = (0.005, 0.995)

# Split and perturbation settings
VAL_FRAC = 0.20
SPLIT_SEED = 0
DEPTHS = [2, 4]
MAX_SUBGRAPHS = 2500
MIN_CENTER_COUNT = 40
MIN_PAIR_COUNT = 25
SUBGRAPH_SEED = 0
PERTURB_BATCH_SIZE = 128

# Model settings
HIDDEN_DIM = 4 * 64
DROPOUT = 0.10
NORM = "batch"
RESIDUAL = True

# Training settings
LR = 3e-4
BATCH_SIZE = 128
MAX_EPOCHS = 2000
PATIENCE = 30
NUM_WORKERS = 4
EDGE_LOSS_WEIGHT = 0.2
EDGE_LOSS_PARAMS = {
    "weighted": False,
    "alpha": 2.0,
    "normalize_by": "graph_std",
    "clip_weight": 4.0,
}

# Experiment output settings
SAVE_GROUP = "gin_perturbation_analysis"
RESULTS_ROOT = PROJECT_ROOT / "results_experiments" / EXPERIMENT_GROUP
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
SAVE_DIR = RESULTS_ROOT / f"{SAVE_GROUP}_{RUN_TIMESTAMP}"
FIGURES_DIR = SAVE_DIR / "figures"


In [ ]:
def _safe_filename(name):
    text = str(name).strip().replace("/", "_")
    chars = [ch if (ch.isalnum() or ch in "._-") else "_" for ch in text]
    cleaned = "".join(chars).strip("._-")
    while "__" in cleaned:
        cleaned = cleaned.replace("__", "_")
    return cleaned or "figure"


def ensure_figure_dir():
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    return FIGURES_DIR


def save_mpl_figure(fig, name, *, close=False):
    path = ensure_figure_dir() / f"{_safe_filename(name)}.pdf"
    fig.savefig(path, bbox_inches="tight", transparent=True)
    print(f"Saved figure -> {path}")
    if close:
        plt.close(fig)
    return path


def save_current_mpl_figure(name, *, close=False):
    return save_mpl_figure(plt.gcf(), name, close=close)


def save_plotly_figure(fig, name, *, scale=2):
    path = ensure_figure_dir() / f"{_safe_filename(name)}.png"
    try:
        fig.write_image(str(path), scale=scale)
        print(f"Saved Plotly figure -> {path}")
    except Exception as exc:
        print(f"Could not save Plotly figure as PNG at {path}: {exc}")
    return path


In [ ]:
from src.data.io import load_graph_dataset_from_dir, select_graph_targets
from src.data.metadata import (
    add_log_metadata_features,
    attach_metadata_to_graphs,
    fill_missing_metadata_for_group,
    infer_global_dim,
    load_aux_metadata_for_dir,
    load_marker_names_from_dir,
    print_graph_and_metadata_fields,
    promote_metadata_to_graph_tensors,
    snapshot_graph_metadata,
    strip_graph_metadata,
)
from src.data.filters import (
    filter_graphs_by_marker_diversity,
    filter_graphs_by_numeric_metadata,
    filter_graphs_by_sphericity,
)
from src.data.preprocessing import interpolate_target_outliers_from_neighbors
from src.data.splits import graph_metadata_key, train_val_split_graphs
from src.data.target_transforms import AsinhStandardizeTransform, standardize_graph_global_features
from src.models.gnn import GINCurvature
from src.training.loop import TrainConfig, train
from src.training.losses import WeightedLossTerm, edge_loss_term
from src.inference.predict import predict_targets

from src.data.subgraphs import build_ego_subgraphs_for_dataset
from src.data.subgraph_sampling import sample_subgraphs_coverage, print_sampling_summary
from src.analysis.perturbation import compute_perturbation_influence_maps
from src.plotting.influence_maps import plot_influence_heatmap, plot_influence_center_resolved


In [ ]:
data_dir = DATA_ROOT / DATA_SUBDIR
graphs = load_graph_dataset_from_dir(str(data_dir))
print(f"Loaded {len(graphs)} organoids.")

meta = load_aux_metadata_for_dir(str(data_dir))
attach_metadata_to_graphs(graphs, meta)

graphs = select_graph_targets(graphs, target_indices=TARGET_INDICES, inplace=False)
if graphs:
    print("Selected y shape:", tuple(graphs[0].y.shape))

marker_names = load_marker_names_from_dir(str(data_dir))
if marker_names is None:
    n_markers_for_names = int(graphs[0].x.size(1)) if graphs else 0
    marker_names = [f"marker_{i}" for i in range(n_markers_for_names)]
print(f"Loaded {len(marker_names)} markers.")


In [ ]:
print_graph_and_metadata_fields(graphs)


In [ ]:
graphs = fill_missing_metadata_for_group(
    graphs,
    field="complexity",
    fill_value=MISSING_COMPLEXITY_GROUP["fill_value"],
    dataset=MISSING_COMPLEXITY_GROUP["dataset"],
    timepoint=MISSING_COMPLEXITY_GROUP["timepoint"],
)


In [ ]:
graphs, g_spherical = filter_graphs_by_sphericity(
    graphs,
    max_sphericity=SPHERICITY_MAX,
    print_summary=True,
    return_rejected=True,
)

g_spherical = filter_graphs_by_marker_diversity(
    g_spherical,
    min_score=SPHERICAL_MARKER_DIVERSITY_MIN,
    print_summary=True,
)

graphs = filter_graphs_by_numeric_metadata(
    graphs,
    key="complexity",
    min_value=COMPLEXITY_THRESHOLD,
    allow_missing=False,
    inplace=False,
    print_summary=True,
)

graphs = graphs + g_spherical
print(f"After filtering and spherical rescue: {len(graphs)} organoids.")

if INTERPOLATE_TARGET_OUTLIERS:
    graphs, outlier_info = interpolate_target_outliers_from_neighbors(
        graphs,
        target_indices=None,
        clip_quantiles=OUTLIER_CLIP_QUANTILES,
    )
else:
    outlier_info = None


In [ ]:
field_specs = [
    {
        "meta_keys": [
            "log_surface_area",
            "log_volume",
            "log_volume_over_area",
            "log_num_cells",
        ],
        "attr_name": "global_feat",
        "kind": "graph_vector",
        "dtype": torch.float32,
    },
]


In [ ]:
base_train, base_val, split_info = train_val_split_graphs(
    graphs,
    val_frac=VAL_FRAC,
    seed=SPLIT_SEED,
    key_fn=graph_metadata_key,
)

print(f"Split -> train: {len(base_train)} | val: {len(base_val)}")


In [ ]:
def remove_attr_from_graphs(graphs, attr_name):
    graphs_out = [copy.copy(g) for g in graphs]
    for g in graphs_out:
        if hasattr(g, attr_name):
            delattr(g, attr_name)
    return graphs_out


def prepare_graph_variant(train_graphs, val_graphs, *, use_global_features: bool):
    g_train = copy.deepcopy(train_graphs)
    g_val = copy.deepcopy(val_graphs)

    if use_global_features:
        g_train = add_log_metadata_features(g_train, inplace=False)
        g_val = add_log_metadata_features(g_val, inplace=False)
        g_train = promote_metadata_to_graph_tensors(g_train, field_specs, inplace=False)
        g_val = promote_metadata_to_graph_tensors(g_val, field_specs, inplace=False)
    else:
        g_train = remove_attr_from_graphs(g_train, "global_feat")
        g_val = remove_attr_from_graphs(g_val, "global_feat")

    val_meta_lookup = snapshot_graph_metadata(g_val)

    g_train = strip_graph_metadata(g_train, inplace=False)
    g_val = strip_graph_metadata(g_val, inplace=False)

    target_transform = AsinhStandardizeTransform(robust=True).fit(g_train)
    target_transform.transform_graphs(g_train)
    target_transform.transform_graphs(g_val)

    center_global, scale_global = None, None
    if use_global_features:
        center_global, scale_global = standardize_graph_global_features(
            g_train,
            g_val,
            attr_name="global_feat",
            robust=False,
        )

    return {
        "g_train": g_train,
        "g_val": g_val,
        "target_transform": target_transform,
        "center_global": center_global,
        "scale_global": scale_global,
        "val_meta_lookup": val_meta_lookup,
    }


def make_gin_model(g_train, depth):
    n_markers = int(g_train[0].x.size(1))
    global_dim = infer_global_dim(g_train)

    model = GINCurvature(
        n_markers=n_markers,
        global_dim=global_dim,
        hidden_dim=HIDDEN_DIM,
        num_layers=depth,
        dropout=DROPOUT,
        residual=RESIDUAL,
        norm=NORM,
    )
    return model


def make_train_config():
    aux_losses = [
        WeightedLossTerm(
            name="edge",
            fn=edge_loss_term,
            weight=EDGE_LOSS_WEIGHT,
            params=EDGE_LOSS_PARAMS,
        ),
    ]

    return TrainConfig(
        lr=LR,
        batch_size=BATCH_SIZE,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        num_workers=NUM_WORKERS,
        aux_losses=aux_losses,
    )


In [ ]:
variant_data = {
    variant_name: prepare_graph_variant(
        base_train,
        base_val,
        use_global_features=use_global_features,
    )
    for variant_name, use_global_features in FEATURE_VARIANTS.items()
}

for variant_name, pack in variant_data.items():
    gd = infer_global_dim(pack["g_train"])
    print(
        f"{variant_name:>11s} | "
        f"train={len(pack['g_train'])} | val={len(pack['g_val'])} | "
        f"global_dim={gd}"
    )

In [ ]:
trained_models = {}
training_logs = {}
prediction_summaries = {}

for variant_name, pack in variant_data.items():
    for depth in DEPTHS:
        family_key = f"GIN | {variant_name} | depth={depth}"
        print("\n" + "=" * 80)
        print("Training", family_key)

        model = make_gin_model(pack["g_train"], depth=depth)
        cfg = make_train_config()

        model, metrics, history = train(model, pack["g_train"], pack["g_val"], cfg)

        trained_models[family_key] = model
        training_logs[family_key] = {
            "metrics": metrics,
            "history": history,
            "variant_name": variant_name,
            "depth": depth,
        }

        y_true, y_pred, X = predict_targets(
            pack["g_val"],
            model,
            batch_size=BATCH_SIZE,
            target_transform=pack["target_transform"],
        )

        prediction_summaries[family_key] = {
            "y_true_shape": np.shape(y_true),
            "y_pred_shape": np.shape(y_pred),
            "X_shape": np.shape(X) if X is not None else None,
            "variant_name": variant_name,
            "depth": depth,
        }

print("\nFinished training", len(trained_models), "models.")

## Neighborhood perturbation analysis

In [ ]:
perturbation_results = {}
sampling_results = {}
sampled_val_subgraphs_by_key = {}

for family_key, model in trained_models.items():
    parts = family_key.split(" | ")
    _, variant_name, depth_part = parts
    depth = int(depth_part.split("=")[1])

    pack = variant_data[variant_name]

    print("\n" + "=" * 80)
    print("Preparing perturbation analysis for", family_key)

    val_subgraphs = build_ego_subgraphs_for_dataset(
        pack["g_val"],
        num_hops=depth,
        max_centers_per_graph=None,
        seed=SUBGRAPH_SEED,
    )

    sampled_val_subgraphs, sample_info = sample_subgraphs_coverage(
        val_subgraphs,
        marker_names=marker_names,
        k_hops=depth,
        max_subgraphs=MAX_SUBGRAPHS,
        min_center_count=MIN_CENTER_COUNT,
        min_pair_count=MIN_PAIR_COUNT,
        seed=SUBGRAPH_SEED,
    )

    sampled_val_subgraphs_by_key[family_key] = sampled_val_subgraphs
    sampling_results[family_key] = sample_info

    print_sampling_summary(sample_info, marker_names)

    res = compute_perturbation_influence_maps(
        subgraphs=sampled_val_subgraphs,
        model=model,
        marker_names=marker_names,
        k_hops=depth,
        mode="single",
        max_subgraphs=None,
        batch_size=PERTURB_BATCH_SIZE,
    )

    perturbation_results[family_key] = {
        "result": res,
        "variant_name": variant_name,
        "depth": depth,
        "n_sampled_subgraphs": len(sampled_val_subgraphs),
    }

print("\nComputed perturbation results for", len(perturbation_results), "models.")

In [ ]:
# Visualize all perturbation maps
for family_key in perturbation_results:
    res = perturbation_results[family_key]["result"]
    depth = perturbation_results[family_key]["depth"]
    family_slug = f"{family_key}_depth_{depth}"

    fig, axes = plot_influence_heatmap(
        res["delta_mu_total"],
        res["hops"],
        res["marker_names"],
        title=f"{family_key} : Δμ (perturbed - base)",
    )
    save_mpl_figure(fig, f"{family_slug}_delta_mu_total")

    fig, axes = plot_influence_heatmap(
        res["delta_lv_total"],
        res["hops"],
        res["marker_names"],
        title=f"{family_key} : Δlogvar (perturbed - base)",
    )
    save_mpl_figure(fig, f"{family_slug}_delta_logvar_total")

    fig, axes = plot_influence_heatmap(
        res["delta_mu_abs_total"],
        res["hops"],
        res["marker_names"],
        title=f"{family_key} : |Δμ|",
    )
    save_mpl_figure(fig, f"{family_slug}_delta_mu_abs_total")

    fig, axes = plot_influence_heatmap(
        res["delta_lv_abs_total"],
        res["hops"],
        res["marker_names"],
        title=f"{family_key} : |Δlogvar|",
    )
    save_mpl_figure(fig, f"{family_slug}_delta_logvar_abs_total")

    fig, axes = plot_influence_center_resolved(
        res["delta_mu_cmarker"],
        res["hops"],
        res["marker_names"],
        title=f"{family_key} : Δμ per perturbed marker conditioned on center marker",
        sort_center=False,
        center_zero=True,
        cmap="RdBu_r",
    )
    save_mpl_figure(fig, f"{family_slug}_delta_mu_center_resolved")

    fig, axes = plot_influence_center_resolved(
        res["delta_lv_cmarker"],
        res["hops"],
        res["marker_names"],
        title=f"{family_key} : Δlogvar per perturbed marker conditioned on center marker",
        sort_center=False,
        center_zero=True,
        cmap="RdBu_r",
    )
    save_mpl_figure(fig, f"{family_slug}_delta_logvar_center_resolved")

    plt.show()


In [ ]:
def perturbation_summary_row(family_key, entry):
    res = entry["result"]
    return {
        "family": family_key,
        "variant_name": entry["variant_name"],
        "depth": entry["depth"],
        "n_sampled_subgraphs": entry["n_sampled_subgraphs"],
        "mean_abs_delta_mu": float(np.nanmean(np.abs(res["delta_mu_total"]))),
        "mean_abs_delta_lv": float(np.nanmean(np.abs(res["delta_lv_total"]))),
        "max_abs_delta_mu": float(np.nanmax(np.abs(res["delta_mu_total"]))),
        "max_abs_delta_lv": float(np.nanmax(np.abs(res["delta_lv_total"]))),
    }

summary_rows = [
    perturbation_summary_row(family_key, entry)
    for family_key, entry in perturbation_results.items()
]

summary_rows = sorted(summary_rows, key=lambda d: (d["variant_name"], d["depth"]))
summary_rows

In [ ]:
def _jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): _jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_jsonable(v) for v in obj]
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, torch.dtype):
        return str(obj)
    if isinstance(obj, np.dtype):
        return str(obj)
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu().tolist()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


def save_joint_experiment(save_dir, config, results_by_experiment, notes=None):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    with open(save_dir / "config.json", "w") as f:
        json.dump(_jsonable(config), f, indent=2)

    with open(save_dir / "results.pkl", "wb") as f:
        pickle.dump(results_by_experiment, f)

    meta = {
        "timestamp": datetime.now().isoformat(),
        "notes": notes,
        "experiments": list(results_by_experiment.keys()),
    }
    with open(save_dir / "meta.json", "w") as f:
        json.dump(_jsonable(meta), f, indent=2)

    print(f"Saved joint experiment -> {save_dir}")

In [ ]:
save_dir = SAVE_DIR

joint_config = {
    "save_group": SAVE_GROUP,
    "data_subdir": DATA_SUBDIR,
    "complexity_threshold": COMPLEXITY_THRESHOLD,
    "field_specs": field_specs,
    "val_frac": VAL_FRAC,
    "split_seed": SPLIT_SEED,
    "feature_variants": FEATURE_VARIANTS,
    "depths": DEPTHS,
    "hidden_dim": HIDDEN_DIM,
    "dropout": DROPOUT,
    "norm": NORM,
    "residual": RESIDUAL,
    "train_config": {
        "lr": LR,
        "batch_size": BATCH_SIZE,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "num_workers": NUM_WORKERS,
        "edge_loss_weight": EDGE_LOSS_WEIGHT,
        "edge_loss_params": EDGE_LOSS_PARAMS,
    },
    "perturbation_config": {
        "max_subgraphs": MAX_SUBGRAPHS,
        "min_center_count": MIN_CENTER_COUNT,
        "min_pair_count": MIN_PAIR_COUNT,
        "subgraph_seed": SUBGRAPH_SEED,
        "perturb_batch_size": PERTURB_BATCH_SIZE,
        "mode": "single",
    },
    "n_train_graphs": len(base_train),
    "n_val_graphs": len(base_val),
    "marker_names": list(marker_names),
    "figures_dir": str(FIGURES_DIR),
}

results_by_experiment = {
    "summary_rows": summary_rows,
    "training_logs": training_logs,
    "prediction_summaries": prediction_summaries,
    "sampling_results": sampling_results,
    "perturbation_results": perturbation_results,
}

save_joint_experiment(
    save_dir=save_dir,
    config=joint_config,
    results_by_experiment=results_by_experiment,
    notes=(
        "Neighborhood perturbation analysis for GINCurvature using the "
        "global-feature setting selected by USE_GLOBAL_FEATURES."
    ),
)